# Datamosh Simulator

A simplified, numpy-based simulation of an MPEG-like encoding pipeline, built for *understanding the mechanics* before applying real datamosh techniques to actual video (see `tools/ffmpeg_datamosh.py` for that).

Pipeline simulated here:
1. Load a sequence of frames.
2. Designate the first frame of each GOP as an I-frame (stored fully).
3. Compute simple block-based motion vectors + residuals for P-frames.
4. Strip I-frames after the first and reconstruct — watch the corruption propagate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

BLOCK_SIZE = 16
SEARCH_RANGE = 8

## 1. Synthetic frame sequence

Generate a small synthetic video: a moving gradient blob, so motion vectors have something nontrivial to estimate. Swap this out for real frames loaded via `cv2.VideoCapture` if you want to experiment with footage.

In [ ]:
def make_synthetic_frames(n_frames=24, size=128):
    frames = []
    yy, xx = np.mgrid[0:size, 0:size]
    for i in range(n_frames):
        cx = size * 0.3 + size * 0.4 * (i / n_frames)
        cy = size * 0.5
        dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        frame = np.clip(255 - dist * 4, 0, 255).astype(np.uint8)
        frames.append(np.stack([frame, frame, frame], axis=-1))
    return frames

frames = make_synthetic_frames()
plt.imshow(frames[0])
plt.title('Frame 0')
plt.axis('off')

## 2. Block-based motion estimation

For each macroblock in frame *t*, search a small window in frame *t-1* for the best matching block (lowest sum-of-absolute-differences). This is a toy version of what `mestimate` does in ffmpeg, and conceptually similar to what real H.264 encoders do per macroblock.

In [ ]:
def estimate_motion_vectors(prev_gray, curr_gray, block_size=BLOCK_SIZE, search_range=SEARCH_RANGE):
    h, w = curr_gray.shape
    vectors = {}
    for by in range(0, h - block_size + 1, block_size):
        for bx in range(0, w - block_size + 1, block_size):
            block = curr_gray[by:by + block_size, bx:bx + block_size]
            best_sad = np.inf
            best_dxdy = (0, 0)
            for dy in range(-search_range, search_range + 1):
                for dx in range(-search_range, search_range + 1):
                    ry, rx = by + dy, bx + dx
                    if ry < 0 or rx < 0 or ry + block_size > h or rx + block_size > w:
                        continue
                    ref_block = prev_gray[ry:ry + block_size, rx:rx + block_size]
                    sad = np.abs(block.astype(int) - ref_block.astype(int)).sum()
                    if sad < best_sad:
                        best_sad = sad
                        best_dxdy = (dx, dy)
            vectors[(bx, by)] = best_dxdy
    return vectors

def to_gray(frame):
    return frame[..., 0]

## 3. Encode as I/P frames

GOP structure: every `gop_size`-th frame is an I-frame (stored as-is). Every other frame is stored as motion vectors + residual relative to the *previous reconstructed frame*.

In [ ]:
def encode_gop(frames, gop_size=8):
    encoded = []
    prev_recon = None
    for i, frame in enumerate(frames):
        if i % gop_size == 0:
            encoded.append({'type': 'I', 'data': frame.copy()})
            prev_recon = frame
        else:
            mvs = estimate_motion_vectors(to_gray(prev_recon), to_gray(frame))
            predicted = motion_compensate(prev_recon, mvs)
            residual = frame.astype(int) - predicted.astype(int)
            encoded.append({'type': 'P', 'mvs': mvs, 'residual': residual})
            prev_recon = frame
    return encoded

def motion_compensate(ref_frame, mvs, block_size=BLOCK_SIZE):
    h, w, c = ref_frame.shape
    out = np.zeros_like(ref_frame)
    for (bx, by), (dx, dy) in mvs.items():
        ry, rx = by + dy, bx + dx
        out[by:by + block_size, bx:bx + block_size] = ref_frame[ry:ry + block_size, rx:rx + block_size]
    return out

## 4. Decode normally vs. decode with I-frames stripped

This is the moment of datamosh: when a P-frame's `motion_compensate` step has no valid I-frame anchor to fall back to and instead keeps building on a stale/wrong reconstruction.

In [ ]:
def decode(encoded, strip_i_after_first=False):
    decoded = []
    prev_recon = None
    first_i_seen = False
    for frame_data in encoded:
        if frame_data['type'] == 'I':
            if strip_i_after_first and first_i_seen:
                # Simulate a missing I-frame: keep reusing the last reconstruction
                # instead of resetting to the true keyframe.
                decoded.append(prev_recon.copy())
                continue
            first_i_seen = True
            decoded.append(frame_data['data'])
            prev_recon = frame_data['data']
        else:
            predicted = motion_compensate(prev_recon, frame_data['mvs'])
            recon = np.clip(predicted.astype(int) + frame_data['residual'], 0, 255).astype(np.uint8)
            decoded.append(recon)
            prev_recon = recon
    return decoded

encoded = encode_gop(frames, gop_size=8)
clean = decode(encoded, strip_i_after_first=False)
moshed = decode(encoded, strip_i_after_first=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, idx in enumerate([0, 7, 8, 16]):
    axes[0, col].imshow(clean[idx]); axes[0, col].set_title(f'clean[{idx}]'); axes[0, col].axis('off')
    axes[1, col].imshow(moshed[idx]); axes[1, col].set_title(f'moshed[{idx}]'); axes[1, col].axis('off')
plt.tight_layout()

Frames 8 and 16 are GOP boundaries (I-frames in the clean decode). In the moshed decode, those slots reuse the prior reconstruction instead of resetting — watch how the blob's old position 'bleeds' forward as later P-frames keep applying fresh motion vectors on top of a stale base.

Try changing `gop_size`, swapping in real video frames, or feeding `moshed` frames into `frame_type_analyzer.ipynb`-style analysis to see how GOP size affects the corruption's spread.